# **Symmetry Breaking in the CC-MUCP via Demand-Aware Lexicographic Ordering**

This notebook presents the Combined Cycle Min-up/Min-down Unit Commitment Problem (CC-MUCP) under a component-based formulation, in which each gas turbine (GT) and steam turbine (ST) is modelled as an individual decision unit. We characterise the two-level unit permutation symmetry that arises when identical packages coexist, a wreath product of symmetric groups, and apply a demand-aware lexicographic ordering that breaks it at both the intra-package and inter-package levels, selecting exactly one canonical commitment schedule per orbit.

---

## *Theoretical Framework*

### **Combined Cycle Power Plants**

A combined cycle (CC) power plant recovers the exhaust heat of one or more gas turbines (GTs) through a heat recovery steam generator (HRSG) to drive a steam turbine (ST), achieving substantially higher thermal efficiency than open-cycle (OC) operation, typically reducing specific fuel consumption and marginal production cost per MWh by 35–40%<sup>[[1]](#ref1)</sup>. This efficiency gain introduces operational coupling between components: the ST can only operate if a sufficient number of GTs have been running long enough to heat the HRSG.

To represent this structure in an optimisation framework, each CC plant is modelled as a **package** $k$, consisting of $n_{gt}$ gas turbines and one steam turbine in a $n_{gt} \times 1$ configuration. The set of all packages is $\mathcal{K} = \{1, \ldots, K\}$.

---

### **The Min-Up/Min-Down Unit Commitment Problem (MUCP)**

The Min-Up/Min-Down Unit Commitment Problem (MUCP) is the core combinatorial optimisation problem underlying thermal unit commitment in power systems. Given a discrete time horizon and a set of production units, the objective is to determine which units should be on or off and how much power they should produce at each period so as to meet electricity demand at minimum cost.

A defining feature of the MUCP is the presence of minimum up-time and minimum down-time constraints: once a unit is started, it must remain on for a minimum number of periods, and once shut down, it must remain off for a minimum duration. These temporal constraints introduce strong inter-period dependencies and constitute the main source of combinatorial complexity. Each unit is characterised by production limits, start-up costs, fixed operating costs, and variable generation costs, and must operate within its feasible production range whenever it is on. The demand must be satisfied at every period.

---

### **The Combined Cycle Min-Up/Min-Down Unit Commitment Problem (CC-MUCP)**

The CC-MUCP extends the MUCP<sup>[[2]](#ref2),[[3]](#ref3)</sup> by explicitly modelling the internal structure of combined cycle power plants. Rather than treating each unit as an independent generator, a CC plant is composed of multiple GTs and one ST organised into packages. The decision process therefore involves determining, for each component and time period: commitment and start-up decisions, operating mode of each GT (OC or CC), and production levels of GTs and ST, such that total demand is satisfied at minimum cost.

In addition to the classical MUCP constraints, the CC-MUCP incorporates mode-dependent GT operation, thermal coupling constraints linking GT activity to ST feasibility, and interdependent production limits where ST output depends on the number of GTs operating in CC mode.

The formulation adopted here follows a **component-based** approach<sup>[[4]](#ref4)</sup>, where each GT and ST is treated as an individual decision unit with its own commitment, start-up, operating mode, and production variables. This provides a fine-grained representation of the internal structure of each package and captures the operational flexibility and coupling between components explicitly. Alternative **configuration-based** formulations represent each package as a single unit selecting among predefined operating configurations<sup>[[5]](#ref5)</sup>; while more compact, they offer less structural transparency.


<hr style="border: 3px solid gray;">

### **Mathematical Formulation**

The CC-MUCP is formulated as a mixed-integer linear program (MILP) under the component-based approach, where each GT operates in one of two mutually exclusive modes at each period: **open-cycle (OC)**, producing electricity independently without contributing to the steam cycle, or **combined-cycle (CC)**, routing exhaust heat through the HRSG to enable ST operation. Binary variables capture commitment, start-up, and mode decisions; continuous variables represent production levels. The formulation integrates classical unit commitment constraints with the mode-dependent and coupling relationships specific to combined cycle operation.

#### **Decision Variables**

For each package $k \in \mathcal{K}$, GT $g \in \{1, \ldots, n_{gt}\}$, and period $t \in \mathcal{T} = \{1, \ldots, T\}$:

\begin{alignat*}{2}
x_{k,g}^t &\in \{0,1\}: &\quad& \text{GT } g \text{ of package } k \text{ is on at } t \\
u_{k,g}^t &\in \{0,1\}: &\quad& \text{GT } g \text{ of package } k \text{ starts up at } t \\
y_{k,g,\mathrm{OC}}^t &\in \{0,1\}: &\quad& \text{GT operates in open cycle at } t \\
y_{k,g,\mathrm{CC}}^t &\in \{0,1\}: &\quad& \text{GT operates in combined cycle at } t \\
p_{k,g,\mathrm{OC}}^t &\in \mathbb{R}_{\geq 0}: &\quad& \text{GT production in open cycle (MW)} \\
p_{k,g,\mathrm{CC}}^t &\in \mathbb{R}_{\geq 0}: &\quad& \text{GT production in combined cycle (MW)}
\end{alignat*}

For the steam turbine of package $k$:

\begin{alignat*}{2}
x_{k,v}^t &\in \{0,1\}: &\quad& \text{ST of package } k \text{ is on at } t \\
u_{k,v}^t &\in \{0,1\}: &\quad& \text{ST of package } k \text{ starts up at } t \\
p_{k,v}^t &\in \mathbb{R}_{\geq 0}: &\quad& \text{ST production (MW)}
\end{alignat*}

---

#### **Parameters**

**Per GT** (same for all GTs in all packages, since packages are identical):
$P_{\min}^{\mathrm{OC}}, P_{\max}^{\mathrm{OC}}$, $P_{\min}^{\mathrm{CC}}, P_{\max}^{\mathrm{CC}}$: production limits by mode (MW); $c_f^{\mathrm{OC}}, c_p^{\mathrm{OC}}, c_f^{\mathrm{CC}}, c_p^{\mathrm{CC}}$: fixed and variable costs by mode; $c_0^g$: start-up cost; $L_g, \ell_g$: minimum up/down times.

**Per ST**: $P_{\min}^v$: minimum production (MW); $\alpha$: MW contributed to ST capacity per GT operating in combined cycle; $c_f^v, c_p^v, c_0^v$: costs; $L_v, \ell_v$: minimum up/down times.

**Coupling per package**: $m_k$: minimum number of GTs on to operate the ST; $\delta^{\mathrm{up}}$: periods GTs must be on before ST can start; $\delta^{\mathrm{dn}}$: periods GTs must remain on after ST shuts down.

---

#### **MILP Formulation**

**Objective** — minimise total operating cost:

\begin{equation*}
\min \sum_{t \in \mathcal{T}} \sum_{k \in \mathcal{K}} \left[
\sum_{g=1}^{n_{gt}} \!\left(
c_f^{\mathrm{OC}} y_{k,g,\mathrm{OC}}^t + c_f^{\mathrm{CC}} y_{k,g,\mathrm{CC}}^t
+ c_0^g u_{k,g}^t
+ c_p^{\mathrm{OC}} p_{k,g,\mathrm{OC}}^t + c_p^{\mathrm{CC}} p_{k,g,\mathrm{CC}}^t
\right)
+ c_f^v x_{k,v}^t + c_0^v u_{k,v}^t + c_p^v p_{k,v}^t
\right]
\end{equation*}

**Individual GT constraints** (for each package $k$, GT $g$, period $t$, with $x_{k,g}^0 = 0$):

\begin{align*}
&\textbf{(SU-GT)} && u_{k,g}^t \geq x_{k,g}^t - x_{k,g}^{t-1} \\
&\textbf{(MU-GT)} && x_{k,g}^{t+s} \geq u_{k,g}^t, \quad s = 1,\ldots,\min(L_g-1,\,T-t) \\
&\textbf{(MD-GT)} && x_{k,g}^{t+s} \leq 1-(x_{k,g}^{t-1}-x_{k,g}^t), \quad t\geq 2,\; s=1,\ldots,\min(\ell_g-1,\,T-t) \\
&\textbf{(MOD)}   && x_{k,g}^t = y_{k,g,\mathrm{OC}}^t + y_{k,g,\mathrm{CC}}^t \\
&\textbf{(PB-OC)} && P_{\min}^{\mathrm{OC}}\, y_{k,g,\mathrm{OC}}^t \leq p_{k,g,\mathrm{OC}}^t \leq P_{\max}^{\mathrm{OC}}\, y_{k,g,\mathrm{OC}}^t \\
&\textbf{(PB-CC)} && P_{\min}^{\mathrm{CC}}\, y_{k,g,\mathrm{CC}}^t \leq p_{k,g,\mathrm{CC}}^t \leq P_{\max}^{\mathrm{CC}}\, y_{k,g,\mathrm{CC}}^t
\end{align*}

**Individual ST constraints** (for each package $k$, period $t$, with $x_{k,v}^0 = 0$):

\begin{align*}
&\textbf{(SU-ST)} && u_{k,v}^t \geq x_{k,v}^t - x_{k,v}^{t-1} \\
&\textbf{(MU-ST)} && x_{k,v}^{t+s} \geq u_{k,v}^t, \quad s=1,\ldots,\min(L_v-1,\,T-t) \\
&\textbf{(MD-ST)} && x_{k,v}^{t+s} \leq 1-(x_{k,v}^{t-1}-x_{k,v}^t), \quad t\geq 2,\; s=1,\ldots,\min(\ell_v-1,\,T-t) \\
&\textbf{(PB-ST)} && P_{\min}^v\, x_{k,v}^t \leq p_{k,v}^t \leq \alpha \sum_{g=1}^{n_{gt}} y_{k,g,\mathrm{CC}}^t
\end{align*}

**Coupling constraints** (for each package $k$, period $t$):

\begin{align*}
&\textbf{(CC1)} && \textstyle\sum_{g=1}^{n_{gt}} x_{k,g}^t \geq m_k \cdot x_{k,v}^t && \text{min GTs for ST operation} \\
&\textbf{(CC2)} && y_{k,g,\mathrm{CC}}^t \leq x_{k,v}^t && \text{CC mode only if ST is on} \\
&\textbf{(CC3)} && y_{k,g,\mathrm{OC}}^t \leq 1 - x_{k,v}^t && \text{OC mode only if ST is off} \\
&\textbf{(CC4)} && m_k \cdot u_{k,v}^t \leq \textstyle\sum_{g=1}^{n_{gt}} x_{k,g}^{t-\delta^{\mathrm{up}}}, \quad t > \delta^{\mathrm{up}} && \text{ST start requires GTs warmed up} \\
& && u_{k,v}^t = 0, \quad t \leq \delta^{\mathrm{up}} && \text{ST cannot start before GTs are ready} \\
&\textbf{(CC5)} && x_{k,g}^{t+s} \geq x_{k,v}^{t-1} - x_{k,v}^t, \quad s=0,\ldots,\delta^{\mathrm{dn}}-1 && \text{GTs stay on after ST shutdown}
\end{align*}


**Global demand constraint:**

\begin{equation*}
\textbf{(D)} \quad \sum_{k \in \mathcal{K}} \left[ \sum_{g=1}^{n_{gt}} \left( p_{k,g,\mathrm{OC}}^t + p_{k,g,\mathrm{CC}}^t \right) + p_{k,v}^t \right] \geq D_t, \quad \forall t \in \mathcal{T}
\end{equation*}

<div style="font-size: 0.8em;">

> **Note on formulation sources.** The GT and ST constraints follow the $(x, u, p)$ formulation of Rajan & Takriti<sup>[[2]](#ref2)</sup> as analysed in Bendotti et al.<sup>[[3]](#ref3)</sup>, and the coupling constraints (CC1–CC5) are based on the component-based model of Liu et al.<sup>[[4]](#ref4)</sup>. Two adaptations are introduced: **(CC4)** is reformulated as $m_k \cdot u_{k,v}^t \leq \sum_g x_{k,g}^{t-\delta^{\mathrm{up}}}$ to avoid requiring a specific designated subset of GTs for ST start-up; and the ST production upper bound in **(PB-ST)** uses $\alpha \sum_g y_{k,g,\mathrm{CC}}^t$, which is exact given that (CC2) and (CC3) force $y_{k,g,\mathrm{CC}}^t = x_{k,g}^t$ when the ST is on and $y_{k,g,\mathrm{CC}}^t = 0$ otherwise.

</div>

<hr style="border: 3px solid gray;">

###  **Symmetry Structure**
#### **Wreath Product Symmetry**

When all $K$ packages are identical and all $n_{gt}$ GTs within each package are identical, the CC-MUCP exhibits a two-level permutation symmetry.

**Intra-package symmetry.** Within each package $k$, the $n_{gt}$ GTs are interchangeable: any permutation $\sigma \in S_{n_{gt}}$ of the GT indices yields a feasible solution of equal cost, since all GT parameters are identical and the coupling constraints (CC1–CC5) depend only on aggregate GT quantities.

**Inter-package symmetry.** The $K$ identical packages are themselves interchangeable as complete blocks: any permutation $\pi \in S_K$ of the package indices yields a feasible solution of equal cost.

These two levels combine into the **wreath product** symmetry group $G = S_{n_{gt}} \wr S_K = \left(S_{n_{gt}}\right)^K \rtimes S_K$, of order $|G| = \left(n_{gt}!\right)^K \cdot K!$. For our example with $n_{gt} = 3$ and $K = 2$, this gives $|G| = (3!)^2 \cdot 2! = 72$: each structurally distinct optimal commitment pattern may generate up to 72 equivalent solutions in the search space.

---

#### **Demand-Aware Lexicographic Ordering**

Let $\pi: \{1,\ldots,T\} \to \{1,\ldots,T\}$ be the permutation ranking periods by decreasing demand: $D_{\pi(1)} \geq D_{\pi(2)} \geq \cdots \geq D_{\pi(T)}$. Define the demand-reordered GT schedule of unit $g$ in package $k$ as $\tilde{x}_{k,g} = (x_{k,g}^{\pi(1)}, \ldots, x_{k,g}^{\pi(T)})$, and the demand-reordered aggregate package schedule as $\tilde{A}_k = \left(\sum_g x_{k,g}^{\pi(1)}, \ldots, \sum_g x_{k,g}^{\pi(T)}\right)$.

Symmetry is broken at two levels:

**GT-level ordering** (breaks $S_{n_{gt}}$ within each package $k$):

\begin{equation*}
\sum_{r=1}^{R} x_{k,g}^{\pi(r)} \geq \sum_{r=1}^{R} x_{k,g+1}^{\pi(r)},
\quad \forall k \in \mathcal{K},\; g = 1,\ldots,n_{gt}-1,\; R = 1,\ldots,T
\end{equation*}

Within each package, the GT covering more of the highest-demand periods receives the lower index.

**Package-level ordering** (breaks $S_K$ across packages):

\begin{equation*}
\sum_{g=1}^{n_{gt}} \sum_{r=1}^{R} x_{k,g}^{\pi(r)} \geq \sum_{g=1}^{n_{gt}} \sum_{r=1}^{R} x_{k+1,g}^{\pi(r)},
\quad \forall k = 1,\ldots,K-1,\; R = 1,\ldots,T
\end{equation*}

The package whose GTs collectively cover more of the highest-demand periods receives the lower package index. Together, both levels assign lower indices to the most heavily loaded components during system stress periods, mirroring the merit-order dispatch convention used in power systems operations.

<hr style="border: 3px solid gray;">

 ### **References**

<a id="ref1"></a>
<sup>[[1]](#ref1)</sup> R. Kehlhofer, J. Warner, H. Nielsen, and T. Bachmann, *Combined-Cycle Gas and Steam Turbine Power Plants*, 3rd ed., PennWell, 2009.

<a id="ref2"></a>
<sup>[[2]](#ref2)</sup> Rajan, D., & Takriti, S. (2005). *Minimum up/down polytopes of the unit commitment problem with start-up costs*. IBM Research Report RC23628.

<a id="ref3"></a>
<sup>[[3]](#ref3)</sup> Bendotti, P., Fouilhoux, P., & Rottner, C. (2018). The min-up/min-down unit commitment polytope. *Journal of Combinatorial Optimization*, 36(3), 1024–1058.

<a id="ref4"></a>
<sup>[[4]](#ref4)</sup> Liu, C., Shahidehpour, M., Li, Z., & Wu, L. (2009). Component and mode models for the short-term scheduling of combined-cycle units. *IEEE Transactions on Power Systems*, 24(2), 976–990.

<a id="ref5"></a>
<sup>[[5]](#ref5)</sup> Morales-España, G., Correa-Posada, C. M., & Ramos, A. (2016). Tight and compact MIP formulation of configuration-based combined-cycle units. *IEEE Transactions on Power Systems*, 31(2), 1350–1359.

## *Computational Study*

### **Implementation Overview**

The computational implementation is organized around three components: the model builder, a set of helper functions, and the solution enumerator.

**Model builder.** The function `build_ccmucp_model` constructs the CC-MUCP MILP in JuMP using HiGHS as the solver. It receives all problem parameters and three keyword arguments that control its behavior:

- `symmetry_breaking`: selects the constraint set to apply (`:none`, `:gt_order`, or `:plant_wide`)
- `demand_rank`: the permutation $\pi$ ranking periods by decreasing demand, required when symmetry breaking is active
- `target_cost`: pins the objective to a fixed value, restricting the feasible region to solutions of exactly that cost during enumeration

**Helper functions.** Five auxiliary functions support the analysis:

- `demand_period_ranking`: computes the permutation $\pi$ from the demand vector $D$
- `read_solution`: extracts the binary commitment arrays from the decision variables `x_g` and `x_v` of a solved model
- `reference_schedule`: maps any solution to the canonical representative of its orbit under $S_{n_{gt}} \wr S_K$, applying lexicographic ordering first at the GT level within each package, then at the package level
- `is_reference`: checks whether a given solution already satisfies both ordering conditions, returning a pair of boolean flags `(gt_ordered, pkg_ordered)`
- `show_schedule`: prints the commitment schedule of all GTs and STs across packages in a readable format

**Solution enumerator.** The function `enumerate_ccmucp` finds all optimal schedules achieving a given cost `best_cost` by iteratively solving the model and appending a no-good cut over the joint binary variables $x_g$ and $x_v$ after each solution found. The cut excludes the current solution as a whole, ensuring that each subsequent solve returns a genuinely distinct commitment pattern. Enumeration terminates when the model becomes infeasible or `max_solutions` is reached. The function is compatible with all three symmetry-breaking schemes, producing 108, 8, or 4 solutions depending on the scheme selected.

### **Model Implementation**

In [1]:
using JuMP, HiGHS
println("Packages loaded | Julia ", VERSION)

Packages loaded | Julia 1.11.5


In [2]:
function build_ccmucp_model(K, n_gt, T, D,
                              Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
                              cf_OC, cp_OC, cf_CC, cp_CC, c0_g,
                              L_g, ell_g,
                              Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
                              m_k, delta_up, delta_dn;
                              symmetry_breaking = :none,
                              demand_rank       = nothing,
                              target_cost       = nothing)
    """
    Builds the CC-MUCP MILP model (component-based) using JuMP with HiGHS.

    Topology:
        K      : number of identical packages
        n_gt   : GTs per package (m×1 configuration)
        T      : number of time periods
        D      : demand vector of length T (MW)

    GT parameters (identical across all GTs in all packages):
        Pmin_OC, Pmax_OC : production limits in open cycle (MW)
        Pmin_CC, Pmax_CC : production limits in combined cycle (MW)
        cf_OC, cp_OC     : fixed and variable costs in open cycle
        cf_CC, cp_CC     : fixed and variable costs in combined cycle
        c0_g             : GT start-up cost
        L_g, ell_g       : min-up / min-down time (periods)

    ST parameters (identical across all packages):
        Pmin_v           : ST minimum production (MW)
        alpha            : ST capacity per active GT (MW)
        cf_v, cp_v, c0_v : ST fixed, variable, and start-up costs
        L_v, ell_v       : ST min-up / min-down time (periods)

    Coupling parameters (identical for all packages):
        m_k              : minimum GTs required to operate the ST
        delta_up         : periods GTs must be on before ST can start
        delta_dn         : periods GTs remain on after ST shuts down

    Keyword arguments:
        symmetry_breaking : constraint set to apply
                            :none       — physical constraints only
                            :gt_order   — adds GT-level ordering (Level 1)
                            :plant_wide — adds GT and package ordering (Level 1 + 2)
        demand_rank       : permutation π from demand_period_ranking(), 
                            required when symmetry_breaking != :none
        target_cost       : pins the objective to this value to restrict
                            the search to optimal solutions during enumeration

    Returns: (model, x_g, u_g, y_OC, y_CC, p_OC, p_CC, x_v, u_v, p_v)
    """
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    # GT binary variables
    @variable(model, x_g[1:K, 1:n_gt, 1:T], Bin)     # GT on/off status
    @variable(model, u_g[1:K, 1:n_gt, 1:T], Bin)     # GT start-up indicator
    @variable(model, y_OC[1:K, 1:n_gt, 1:T], Bin)    # GT open-cycle mode
    @variable(model, y_CC[1:K, 1:n_gt, 1:T], Bin)    # GT combined-cycle mode

    # GT integer production variables (MW)
    @variable(model, p_OC[1:K, 1:n_gt, 1:T] >= 0)   # GT production in OC
    @variable(model, p_CC[1:K, 1:n_gt, 1:T] >= 0)   # GT production in CC

    # ST binary variables
    @variable(model, x_v[1:K, 1:T], Bin)              # ST on/off status
    @variable(model, u_v[1:K, 1:T], Bin)              # ST start-up indicator

    # ST integer production variable (MW)
    @variable(model, p_v[1:K, 1:T] >= 0)         # ST production

    # Objective: minimise total operating cost across all packages,
    # GTs, and periods, summing fixed, variable, and start-up costs
    # for both GT modes (OC, CC) and the ST
    @objective(model, Min,
        sum(
            cf_OC * y_OC[k,g,t] + cf_CC * y_CC[k,g,t] +
            c0_g  * u_g[k,g,t]  +
            cp_OC * p_OC[k,g,t] + cp_CC * p_CC[k,g,t]
            for k in 1:K, g in 1:n_gt, t in 1:T) +
        sum(
            cf_v * x_v[k,t] + c0_v * u_v[k,t] + cp_v * p_v[k,t]
            for k in 1:K, t in 1:T))

    # GT constraints
    for k in 1:K, g in 1:n_gt
        # (SU-GT) start-up occurs when GT switches from off to on;
        # initial state assumed off (x_{k,g}^0 = 0)
        @constraint(model, u_g[k,g,1] >= x_g[k,g,1])
        for t in 2:T
            @constraint(model, u_g[k,g,t] >= x_g[k,g,t] - x_g[k,g,t-1])
        end

        for t in 1:T
            # (MU-GT) GT must remain on for at least L_g periods after start-up
            for s in 1:min(L_g - 1, T - t)
                @constraint(model, x_g[k,g,t+s] >= u_g[k,g,t])
            end

            # (MOD) GT is either in open-cycle or combined-cycle mode when on,
            # and in neither mode when off
            @constraint(model, x_g[k,g,t] == y_OC[k,g,t] + y_CC[k,g,t])

            # (PB-OC) production in open-cycle mode is bounded between
            # Pmin_OC and Pmax_OC when active, and zero when inactive
            @constraint(model, p_OC[k,g,t] >= Pmin_OC * y_OC[k,g,t])
            @constraint(model, p_OC[k,g,t] <= Pmax_OC * y_OC[k,g,t])

           # (PB-CC) production in combined-cycle mode is bounded between
            # Pmin_CC and Pmax_CC when active, and zero when inactive
            @constraint(model, p_CC[k,g,t] >= Pmin_CC * y_CC[k,g,t])
            @constraint(model, p_CC[k,g,t] <= Pmax_CC * y_CC[k,g,t])
        end

        for t in 2:T
             # (MD-GT) GT must remain off for at least ell_g periods after shutdown
            for s in 1:min(ell_g - 1, T - t)
                @constraint(model,
                    x_g[k,g,t+s] <= 1 - (x_g[k,g,t-1] - x_g[k,g,t]))
            end
        end
    end

    # ST constraints
    for k in 1:K
        # (SU-ST) start-up occurs when ST switches from off to on;
        # initial state assumed off (x_{k,v}^0 = 0)
        @constraint(model, u_v[k,1] >= x_v[k,1])
        for t in 2:T
            @constraint(model, u_v[k,t] >= x_v[k,t] - x_v[k,t-1])
        end

        for t in 1:T
            # (MU-ST) ST must remain on for at least L_v periods after start-up
            for s in 1:min(L_v - 1, T - t)
                @constraint(model, x_v[k,t+s] >= u_v[k,t])
            end

            # (PB-ST) ST production is at least Pmin_v when on;
            # upper bound depends on the number of GTs in combined-cycle mode,
            # each contributing alpha MW to ST capacity
            @constraint(model, p_v[k,t] >= Pmin_v * x_v[k,t])
            @constraint(model, p_v[k,t] <= alpha * sum(y_CC[k,g,t] for g in 1:n_gt))
        end

        for t in 2:T
            # (MD-ST) ST must remain off for at least ell_v periods after shutdown
            for s in 1:min(ell_v - 1, T - t)
                @constraint(model,
                    x_v[k,t+s] <= 1 - (x_v[k,t-1] - x_v[k,t]))
            end
        end
    end

    # Coupling constraints between GTs and ST within each package
    for k in 1:K, t in 1:T
        # (CC1) at least m_k GTs must be on for the ST to operate
        @constraint(model,
            sum(x_g[k,g,t] for g in 1:n_gt) >= m_k * x_v[k,t])

        for g in 1:n_gt
            # (CC2) a GT can operate in combined-cycle mode only if the ST is on
            @constraint(model, y_CC[k,g,t] <= x_v[k,t])

            # (CC3) a GT can operate in open-cycle mode only if the ST is off
            @constraint(model, y_OC[k,g,t] <= 1 - x_v[k,t])
        end

        # (CC4) ST start-up requires at least m_k GTs to have been on
        # for delta_up periods; ST cannot start in the first delta_up periods
        if t <= delta_up
            @constraint(model, u_v[k,t] == 0)
        else
            @constraint(model,
                m_k * u_v[k,t] <=
                sum(x_g[k,g,t - delta_up] for g in 1:n_gt))
        end

        # (CC5) all GTs must remain on for delta_dn periods after ST shuts down
        if t >= 2
            for s in 0:(delta_dn - 1)
                if t + s <= T
                    for g in 1:n_gt
                        @constraint(model,
                            x_g[k,g,t+s] >= x_v[k,t-1] - x_v[k,t])
                    end
                end
            end
        end
    end

    # Global demand constraint (D)
    # total production from all GTs and STs must meet system demand at every period
    for t in 1:T
        @constraint(model,
            sum(p_OC[k,g,t] + p_CC[k,g,t] for k in 1:K, g in 1:n_gt) +
            sum(p_v[k,t] for k in 1:K) >= D[t])
    end

    # Symmetry-breaking ordering constraints
    # active only when symmetry_breaking != :none;
    # uses prefix sums over demand-ranked periods given by demand_rank
    if symmetry_breaking in (:gt_order, :plant_wide)
        @assert demand_rank !== nothing "demand_rank required for symmetry breaking"
        # Level 1: within each package, GT g must cover at least as many
        # high-demand periods as GT g+1 (intra-package GT ordering)
        for k in 1:K, g in 1:(n_gt-1)
            for R in 1:T
                @constraint(model,
                    sum(x_g[k,g,   demand_rank[r]] for r in 1:R) >=
                    sum(x_g[k,g+1, demand_rank[r]] for r in 1:R))
            end
        end
    end

    if symmetry_breaking == :plant_wide
        # Level 2: package k must cover at least as many high-demand periods
        # as package k+1 in aggregate (inter-package ordering)
        for k in 1:(K-1)
            for R in 1:T
                @constraint(model,
                    sum(x_g[k,   g, demand_rank[r]] for g in 1:n_gt, r in 1:R) >=
                    sum(x_g[k+1, g, demand_rank[r]] for g in 1:n_gt, r in 1:R))
            end
        end
    end

    # Target cost constraint
    # pins the objective to target_cost during enumeration,
    # restricting the search to optimal solutions only
    if target_cost !== nothing
        @constraint(model,
            sum(cf_OC*y_OC[k,g,t] + cf_CC*y_CC[k,g,t] +
                c0_g*u_g[k,g,t]   +
                cp_OC*p_OC[k,g,t] + cp_CC*p_CC[k,g,t]
                for k in 1:K, g in 1:n_gt, t in 1:T) +
            sum(cf_v*x_v[k,t] + c0_v*u_v[k,t] + cp_v*p_v[k,t]
                for k in 1:K, t in 1:T) == target_cost)
    end

    return model, x_g, u_g, y_OC, y_CC, p_OC, p_CC, x_v, u_v, p_v
end

# Helper functions
function demand_period_ranking(D)
    """
    Returns the permutation π ranking time periods from highest to
    lowest demand. Ties broken by period index (earlier first).
    """
    return sortperm(D, rev=true)
end


function read_solution(x_g, x_v, K, n_gt, T)
    # Extracts binary commitment arrays x_g and x_v from a solved JuMP model.
    return round.(Int, value.(x_g)), round.(Int, value.(x_v))
end

function reference_schedule(gt_plan, st_plan, K, n_gt, T, demand_rank)
    """
    Returns the canonical representative of a solution's orbit
    under the wreath product group S_{n_gt} ≀ S_K.

    Level 1 (GT ordering): within each package, GTs are sorted so that
    the unit covering more high-demand periods receives the lower index,
    matching the intra-package symmetry-breaking constraints.

    Level 2 (package ordering): packages are sorted so that the package
    whose GTs collectively cover more high-demand periods receives the
    lower index, matching the inter-package symmetry-breaking constraints.

    Ordering uses prefix sums over demand-ranked periods (demand_rank),
    with lexicographic comparison as tiebreaker.
    """
    gt_new = copy(gt_plan)
    st_new = copy(st_plan)

    # GT-level: sort GTs within each package by prefix sum vector (desc lex)
    for k in 1:K
        prefix_sums = [cumsum(gt_plan[k, g, demand_rank]) for g in 1:n_gt]
        perm = sortperm(prefix_sums, rev=true)
        for g in 1:n_gt
            gt_new[k, g, :] = gt_plan[k, perm[g], :]
        end
    end

    # Package-level: sort packages by aggregate prefix sum vector (desc lex)
    agg_prefix = [cumsum([sum(gt_new[k, g, demand_rank[r]] for g in 1:n_gt)
                          for r in 1:T]) for k in 1:K]
    pkg_perm = sortperm(agg_prefix, rev=true)

    gt_canon = copy(gt_new)
    st_canon = copy(st_new)
    for k in 1:K
        gt_canon[k, :, :] = gt_new[pkg_perm[k], :, :]
        st_canon[k, :]    = st_new[pkg_perm[k], :]
    end

    return gt_canon, st_canon
end

function is_reference(gt_plan, K, n_gt, T, demand_rank)
    # Check whether a solution satisfies GT-level and package-level ordering.
    # Returns (gt_ordered, pkg_ordered).

    gt_ordered = true
    for k in 1:K, g in 1:(n_gt-1)
        for R in 1:T
            if sum(gt_plan[k, g,   demand_rank[r]] for r in 1:R) <
               sum(gt_plan[k, g+1, demand_rank[r]] for r in 1:R)
                gt_ordered = false
                break
            end
        end
        !gt_ordered && break
    end

    pkg_ordered = true
    for k in 1:(K-1)
        for R in 1:T
            if sum(gt_plan[k,   g, demand_rank[r]] for g in 1:n_gt, r in 1:R) <
               sum(gt_plan[k+1, g, demand_rank[r]] for g in 1:n_gt, r in 1:R)
                pkg_ordered = false
                break
            end
        end
        !pkg_ordered && break
    end

    return gt_ordered, pkg_ordered
end

function show_schedule(gt_plan, st_plan, K, n_gt, T, label="")
    # Prints the commitment schedule of all GTs and STs across packages.
    label != "" && println(label)
    for k in 1:K
        println("  Package ", k, ":")
        for g in 1:n_gt
            println("    GT ", g, ": ", gt_plan[k,g,:],
                    "  (hours on: ", sum(gt_plan[k,g,:]), ")")
        end
        println("    ST  : ", st_plan[k,:],
                "  (hours on: ", sum(st_plan[k,:]), ")")
    end
end

function enumerate_ccmucp(K, n_gt, T, D,
                            Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
                            cf_OC, cp_OC, cf_CC, cp_CC, c0_g,
                            L_g, ell_g,
                            Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
                            m_k, delta_up, delta_dn,
                            best_cost;
                            symmetry_breaking = :none,
                            demand_rank       = nothing,
                            max_solutions     = 500)
    # Enumerate all optimal CC-MUCP schedules achieving best_cost.
    solutions = Vector{Tuple{Array{Int,3}, Matrix{Int}}}()

    for _ in 1:max_solutions
        m, x_g, u_g, y_OC, y_CC, p_OC, p_CC, x_v, u_v, p_v =
            build_ccmucp_model(
                K, n_gt, T, D,
                Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
                cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
                Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
                m_k, delta_up, delta_dn;
                symmetry_breaking = symmetry_breaking,
                demand_rank       = demand_rank,
                target_cost       = best_cost)

        for (prev_gt, prev_st) in solutions
            @constraint(m,
                sum(1 - x_g[k,g,t]
                    for k in 1:K, g in 1:n_gt, t in 1:T
                    if prev_gt[k,g,t] == 1; init=0) +
                sum(x_g[k,g,t]
                    for k in 1:K, g in 1:n_gt, t in 1:T
                    if prev_gt[k,g,t] == 0; init=0) +
                sum(1 - x_v[k,t]
                    for k in 1:K, t in 1:T
                    if prev_st[k,t] == 1; init=0) +
                sum(x_v[k,t]
                    for k in 1:K, t in 1:T
                    if prev_st[k,t] == 0; init=0) >= 1)
        end

        optimize!(m)
        termination_status(m) != OPTIMAL && break
        gt_plan, st_plan = read_solution(x_g, x_v, K, n_gt, T)
        push!(solutions, (gt_plan, st_plan))
    end

    return solutions
end

enumerate_ccmucp (generic function with 1 method)

### **Example**: Two-Package CC-MUCP Instance

We consider a CC plant with $K = 2$ identical packages, each consisting of $n_{gt} = 3$ gas turbines and one steam turbine ($3 \times 1$ configuration), over a weekly horizon of $T = 21$ periods structured as three load levels per day (Low, Mid, High).

GT and ST parameters are reported in Tables 1a and 1b. In combined-cycle mode, GTs operate at lower fixed and variable costs than in open-cycle mode, reflecting the efficiency gain from heat recovery. The ST incurs no variable cost ($c_p^v = 0$) since it runs on residual exhaust heat, and its higher start-up cost ($c_0^v = 15{,}000$ \$) and longer minimum up/down times ($L_v = 3$, $\ell_v = 2$) reflect the thermal inertia of the steam cycle. Coupling parameters $m_k = 2$, $\delta^{\mathrm{up}} = 1$, and $\delta^{\mathrm{dn}} = 1$ enforce that at least two GTs must be online before the ST can start, and must remain so after it shuts down.

<div align="center">
<table style="border: none; border-collapse: collapse;">
<tr valign="top">
<td align="center" style="border: none;">

<sub><b>Table 1a.</b> Gas turbine parameters for open-cycle and combined-cycle operation.</sub>


$$
\begin{array}{lcc}
\hline
\textbf{Parameter} & \textbf{Open Cycle} & \textbf{Combined Cycle} \\
\hline
\hline
P_{\max} \, (\text{MW}) & 150 & 145 \\
P_{\min} \, (\text{MW}) & 50 & 60 \\
c_f \, (\$/\text{h}) & 844 & 549 \\
c_p \, (\$/\text{MWh}) & 37.1 & 24.1 \\
c_0 \, (\$/\text{start}) & 25{,}000 & \text{--} \\
\text{Min-up } L_g \, (\text{h}) & 2 & \text{--} \\
\text{Min-down } \ell_g \, (\text{h}) & 1 & \text{--} \\
\hline
\end{array}
$$


</td>
<td width="60" style="border: none;"></td>
<td align="center" style="border: none;">

<sub><b>Table 1b.</b> Steam turbine parameters.</sub>


$$
\begin{array}{lc}
\hline
\textbf{Parameter} & \textbf{Value} \\
\hline
\hline
P_{\min}^v \, (\text{MW}) & 20 \\
\alpha \, (\text{MW/GT}) & 40 \\
c_f^v \, (\$/\text{h}) & 42 \\
c_p^v \, (\$/\text{MWh}) & 0 \\
c_0^v \, (\$/\text{start}) & 15{,}000 \\
\text{Min-up } L_v \, (\text{h}) & 3 \\
\text{Min-down } \ell_v \, (\text{h}) & 2 \\
\hline
\end{array}
$$


</td>
</tr>
</table>
</div>

The demand profile is reported in Table 2. With a total system capacity of $K \cdot (n_{gt} \cdot P_{\max}^{\mathrm{CC}} + n_{gt} \cdot \alpha) = 1{,}110$ MW, peak periods require near-full utilization of both packages, while low-demand periods can be covered by a reduced subset of units. The demand ranking $\pi$, computed by `demand_period_ranking`, orders periods from highest to lowest demand:

$$\pi = [15, 9, 12, 3, 6, 18, 21, 14, 17, 20, 8, 11, 2, 5, 16, 19, 13, 7, 1, 10, 4]$$

and is used by the symmetry-breaking constraints and `reference_schedule` to assign lower indices to units most active during system stress. Under the assumed parameter symmetry, the problem exhibits a wreath product symmetry group $S_3 \wr S_2$ of order $(3!)^2 \cdot 2! = 72$.

In [3]:
# Topology
K     = 2          # number of identical packages
n_gt  = 3          # 3×1 configuration
T     = 21         # 7 days × 3 periods/day

# GT parameters
Pmax_OC = 150.0    # MW — max production in open cycle
Pmin_OC =  50.0    # MW — min production in open cycle
Pmax_CC = 145.0    # MW — max production in combined cycle
Pmin_CC =  60.0    # MW — min production in combined cycle
cf_OC   = 844.0    # $/h — fixed cost open cycle
cp_OC   =  37.1    # $/MWh — variable cost open cycle
cf_CC   = 549.0    # $/h — fixed cost combined cycle
cp_CC   =  24.1    # $/MWh — variable cost combined cycle
c0_g    = 25_000.0 # $ — GT start-up cost
L_g     = 2        # h — GT minimum up time
ell_g   = 1        # h — GT minimum down time

# ST parameters
Pmin_v  =  20.0    # MW — ST minimum production
alpha   =  40.0    # MW — ST capacity contribution per active GT in CC mode
cf_v    =  42.0    # $/h — ST fixed cost
cp_v    =   0.0    # $/MWh — ST variable cost (residual heat, no fuel)
c0_v    = 15_000.0 # $ — ST start-up cost
L_v     = 3        # h — ST minimum up time
ell_v   = 2        # h — ST minimum down time

# Coupling parameters
m_k      = 2       # min GTs required to operate the ST
delta_up = 1       # h — GTs must be on before ST can start
delta_dn = 1       # h — GTs remain on after ST shuts down

# Demand profile: weekly structure with three load levels per day
# (Low, Mid, High) repeated across 7 days
DAYS   = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
LEVELS = ["L", "M", "H"]

# period label for period t: e.g. t=1 -> "Mon-Low", t=3 -> "Mon-High"
period_label(t) = DAYS[(t-1)÷3 + 1] * "-" * LEVELS[(t-1)%3 + 1]

D = [420.,620.,900., 400.,600.,880., 430.,640.,920.,
     410.,630.,910., 450.,700.,1000., 520.,680.,820.,
     500.,650.,780.]

demand_rank = demand_period_ranking(D)

println("Topology: K = ", K, " packages | ", n_gt, "×1 configuration | T = ", T, " periods")
println("Max system capacity: ", K * (n_gt * Pmax_CC + n_gt * alpha), " MW")

println("Peak demand:    ", maximum(D), " MW at t = ", argmax(D),
        " (", period_label(argmax(D)), ")")
println("Minimum demand: ", minimum(D), " MW at t = ", argmin(D),
        " (", period_label(argmin(D)), ")")

println("\nDemand profile:")
for t in 1:T
    bar = repeat("█", round(Int, D[t]/25))
    println("  t = ", lpad(t, 2), " | ",
        rpad(period_label(t), 10),
        " | ", lpad(Int(D[t]), 4), " MW | ", bar)
end

println("\nRanking π: ", demand_rank)
println("Symmetry group: S_", n_gt, " ≀ S_", K,
        "  |  Order = ", factorial(n_gt)^K * factorial(K))

Topology: K = 2 packages | 3×1 configuration | T = 21 periods
Max system capacity: 1110.0 MW
Peak demand:    1000.0 MW at t = 15 (Fri-H)
Minimum demand: 400.0 MW at t = 4 (Tue-L)

Demand profile:
  t =  1 | Mon-L      |  420 MW | █████████████████
  t =  2 | Mon-M      |  620 MW | █████████████████████████
  t =  3 | Mon-H      |  900 MW | ████████████████████████████████████
  t =  4 | Tue-L      |  400 MW | ████████████████
  t =  5 | Tue-M      |  600 MW | ████████████████████████
  t =  6 | Tue-H      |  880 MW | ███████████████████████████████████
  t =  7 | Wed-L      |  430 MW | █████████████████
  t =  8 | Wed-M      |  640 MW | ██████████████████████████
  t =  9 | Wed-H      |  920 MW | █████████████████████████████████████
  t = 10 | Thu-L      |  410 MW | ████████████████
  t = 11 | Thu-M      |  630 MW | █████████████████████████
  t = 12 | Thu-H      |  910 MW | ████████████████████████████████████
  t = 13 | Fri-L      |  450 MW | ██████████████████
  t = 14 | Fri-M     

In [4]:
# Load previously saved results
using JLD2

results_file = "ccmucp_results.jld2"

if isfile(results_file)
    @load results_file sols_base sols_gt sols_full best_cost demand_rank
    println("Results loaded from ", results_file)
    println("  Baseline:      ", length(sols_base), " solutions")
    println("  GT-ordered:    ", length(sols_gt),   " solutions")
    println("  Plant-wide:    ", length(sols_full),  " solutions")
    println("  Optimal cost:  \$", round(best_cost, digits=2))
else
    println("No saved results found at ", results_file)
    println("Run next Cell to solve from scratch.")
end

Results loaded from ccmucp_results.jld2
  Baseline:      108 solutions
  GT-ordered:    8 solutions
  Plant-wide:    4 solutions
  Optimal cost:  $489683.0


#### **Optimal Cost and Solution Enumeration**

The optimal cost is first obtained by solving the baseline model without symmetry breaking. The three enumeration runs: baseline (`:none`), GT-level ordering (`:gt_order`), and plant-wide ordering (`:plant_wide`), then find all distinct optimal schedules at that cost, with each run appending no-good cuts to exclude previously found solutions. Results are saved to `ccmucp_results.jld2` for subsequent analysis.

In [5]:
# Find optimal cost
m0, xg0, ug0, yOC0, yCC0, pOC0, pCC0, xv0, uv0, pv0 =
    build_ccmucp_model(
        K, n_gt, T, D,
        Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
        cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
        Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
        m_k, delta_up, delta_dn;
        symmetry_breaking = :none)
optimize!(m0)
best_cost = objective_value(m0)
println("Optimal cost: \$", round(best_cost, digits=2))

# Enumerate under each symmetry-breaking scheme
println("\nEnumerating optimal solutions")

sols_base = enumerate_ccmucp(
    K, n_gt, T, D,
    Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
    cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
    Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
    m_k, delta_up, delta_dn, best_cost;
    symmetry_breaking=:none)

sols_gt = enumerate_ccmucp(
    K, n_gt, T, D,
    Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
    cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
    Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
    m_k, delta_up, delta_dn, best_cost;
    symmetry_breaking=:gt_order, demand_rank=demand_rank)

sols_full = enumerate_ccmucp(
    K, n_gt, T, D,
    Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
    cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
    Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
    m_k, delta_up, delta_dn, best_cost;
    symmetry_breaking=:plant_wide, demand_rank=demand_rank)

println("Baseline (no symmetry breaking):   ", length(sols_base), " solutions")
println("GT-level ordering (:gt_order):     ", length(sols_gt),   " solutions")
println("Plant-wide ordering (:plant_wide): ", length(sols_full),  " solutions")

    using JLD2
    jldsave("ccmucp_results.jld2";
        sols_base, sols_gt, sols_full,
        best_cost, demand_rank,
        K, n_gt, T, D,
        Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
        cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
        Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
        m_k, delta_up, delta_dn)

    println("\nResults saved: ", length(sols_base), " baseline, ",
            length(sols_gt), " gt-ordered, ",
            length(sols_full), " plant-wide solutions.")

Optimal cost: $489683.0

Enumerating optimal solutions
Baseline (no symmetry breaking):   108 solutions
GT-level ordering (:gt_order):     8 solutions
Plant-wide ordering (:plant_wide): 4 solutions

Results saved: 108 baseline, 8 gt-ordered, 4 plant-wide solutions.


In [6]:
function render_bar(vec::AbstractVector{<:Integer})
    first_on = findfirst(==(1), vec)
    cells    = String[]
    for t in 1:length(vec)
        if vec[t] == 1;                              push!(cells, "█ ")
        elseif isnothing(first_on) || t < first_on; push!(cells, "← ")
        else;                                        push!(cells, "→ ")
        end
    end
    return lpad(sum(vec), 2) * "h", join(cells)
end

function period_header(prefix_len::Int)
    day_row = level_row = repeat(" ", prefix_len)
    for t in 1:T
        (t-1) % 3 == 0 && (day_row *= " " * rpad(DAYS[(t-1)÷3 + 1], 5))
        level_row *= LEVELS[(t-1)%3 + 1] * " "
    end
    return day_row, level_row
end

function print_orbit_schedules(orbit_entries, T, n_gt, K, group_ord)
    sep            = repeat("─", 18 + T * 2)
    day_row, level_row = period_header(18)

    println("  █ = ON   ← = before start   → = after end\n")

    for (i, entry) in enumerate(orbit_entries)
        ref_gt, ref_st = entry.first
        n_sols = length(entry.second)
        println(sep)
        println("  Orbit $i  —  $n_sols solutions  (|Stab| = $(group_ord ÷ n_sols))")
        println(sep)
        println(day_row)
        println(level_row)
        for k in 1:K
            println("  Package $k:")
            for g in 1:n_gt
                h_str, bar = render_bar(ref_gt[k, g, :])
                println("    GT $g  ($h_str):  $bar")
            end
            h_str, bar = render_bar(ref_st[k, :])
            println("    ST    ($h_str):  $bar")
            k < K && println()
        end
        println()
    end
    println(sep)
end

group_ord = factorial(n_gt)^K * factorial(K)
cost_fmt  = "\$" * replace(string(round(Int, best_cost)), r"(?<=\d)(?=(\d{3})+$)" => ",")

# Group solutions into orbits under the wreath product symmetry G = S_{n_gt} ≀ S_K
orbit_map = Dict{Tuple{Array{Int,3}, Matrix{Int}}, Vector{Int}}()
for (idx, (gt_plan, st_plan)) in enumerate(sols_base)
    ref_gt, ref_st = reference_schedule(gt_plan, st_plan, K, n_gt, T, demand_rank)
    key = (ref_gt, ref_st)
    haskey(orbit_map, key) ? push!(orbit_map[key], idx) : (orbit_map[key] = [idx])
end

n_orbits   = length(orbit_map)
sym_status = length(sols_full) == n_orbits ? "Full symmetry breaking achieved" :
                                             "Partial symmetry breaking"

println("  Symmetry reduction:  $(length(sols_base)) solutions  →  $(length(sols_gt)) (GT ordering)  →  $(length(sols_full)) (plant-wide)  |  |G| = $group_ord")
println("  Optimal solution:    Cost: $cost_fmt  ·  $n_orbits orbits  ·  $sym_status")

orbit_entries = sort(collect(orbit_map), by = x -> first(sort(x.second)))
print_orbit_schedules(orbit_entries, T, n_gt, K, group_ord)

  Symmetry reduction:  108 solutions  →  8 (GT ordering)  →  4 (plant-wide)  |  |G| = 72
  Optimal solution:    Cost: $489,683  ·  4 orbits  ·  Full symmetry breaking achieved
  █ = ON   ← = before start   → = after end

────────────────────────────────────────────────────────────
  Orbit 1  —  18 solutions  (|Stab| = 4)
────────────────────────────────────────────────────────────
                   Mon   Tue   Wed   Thu   Fri   Sat   Sun  
                  L M H L M H L M H L M H L M H L M H L M H 
  Package 1:
    GT 1  (21h):  █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
    GT 2  (21h):  █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
    GT 3  (20h):  ← █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
    ST    (20h):  ← █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 

  Package 2:
    GT 1  (21h):  █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
    GT 2  (21h):  █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
    GT 3  ( 2h):  ← ← ← ← ← ← ← ← ← ← ← ← ← █ █ → → → → → → 
    ST    (20h):  ← █ █ █ █ █ █ █ █ █ █ 

#### **Results Analysis**

The baseline enumeration yields 108 optimal commitment schedules, all achieving a minimum cost of \$489,683. This multiplicity is entirely explained by symmetry: identical GTs within each package and identical packages within the plant generate multiple labelings of the same physical operation.

**Orbit structure and symmetry decomposition.**

The 108 solutions distribute across exactly 4 orbits under the wreath product group of order $|G| = (3!)^2 \cdot 2! = 72$, each grouping operationally identical schedules that differ only in unit labeling.

<div align="center">

<sub><b>Table 3.</b> Orbit decomposition under $S_3 \wr S_2$, $|G| = 72$. Each orbit satisfies $|\mathcal{O}| \cdot |\mathrm{Stab}| = |G|$.</sub>

\begin{array}{c|c|c}
\hline
\textbf{Orbit} & \lvert\mathcal{O}\rvert & \lvert\mathrm{Stab}\rvert \\
\hline
\hline
1 & 18 & 4 \\
2 & 18 & 4 \\
3 & 36 & 2 \\
4 & 36 & 2 \\
\hline
\textbf{Total} & 108 &  \\
\hline
\end{array}

</div>

The partition is exact ($18 + 18 + 36 + 36 = 108$), with every orbit size dividing $|G| = 72$, consistent with the orbit-stabilizer relation $|\mathcal{O}| \cdot |\mathrm{Stab}| = |G|$. Smaller orbits correspond to larger stabilizers, indicating schedules with higher internal symmetry; larger orbits arise when fewer permutations leave the schedule invariant, reflecting more differentiated unit behavior.

**Stabilizer interpretation.**

In Orbits 1 and 2 ($|\mathrm{Stab}| = 4$), GT 1 and GT 2 of Package 2 follow identical 21-period schedules and are fully interchangeable, producing a larger stabilizer. In Orbits 3 and 4 ($|\mathrm{Stab}| = 2$), each GT occupies a distinct temporal role across the horizon, leaving fewer symmetry-preserving permutations.

**Effect of the symmetry-breaking levels.**

GT-level ordering reduces the solution set from 108 to 8 by enforcing that units covering more high-demand periods receive lower indices within each package. Package-level ordering further reduces it to 4 by ensuring the package contributing more during peak demand is consistently labeled Package 1. Together, both levels retain exactly one representative per orbit, confirming that the reduction is complete.

**Physical interpretation of the reference schedules.**

The four reference schedules share a common backbone: GT 1 runs for all 21 periods in every package and orbit, anchoring the minimum system demand of 400 MW at $t = 4$. GT 2 follows the same full-horizon commitment in Orbits 1 and 2, but operates for fewer periods in Orbits 3 and 4 (20h in Orbit 3, Package 1; 15h in Orbit 4, Package 2), reflecting the more differentiated structure of those orbits. GT 3 is the most flexible unit: in Package 1 it activates early (from $t = 1$ in Orbit 3, from $t = 2$ in the others) to cover sustained mid- and high-demand periods, while in Package 2 it activates later at $t = 14$ to respond to the global peak at $t = 15$, remaining active through $t = 21$ in Orbits 2, 3, and 4, and shutting down after just 2 periods in Orbit 1.

The steam turbines in both packages operate for 20 periods across all four reference schedules, starting at $t = 2$, one period after the minimum number of GTs are committed, consistent with the warm-up constraint $\delta^{\mathrm{up}} = 1$.

## **Discussion**

The 108 optimal schedules of the baseline enumeration collapse into only 4 genuinely distinct operational patterns, with the orbit structure clarifying how they differ. Orbits of size 18 correspond to schedules where multiple GTs follow identical commitment trajectories and are fully interchangeable, while orbits of size 36 reflect more differentiated unit behavior, where each GT occupies a distinct temporal role. In operational terms, this distinction captures whether units act as redundant capacity fulfilling the same role or as differentiated resources with specialized activation patterns.

The demand-aware ordering resolves this labeling ambiguity by linking unit indices to their contribution during high-demand periods. Since the system peak occurs at $t = 15$ (1000 MW), followed by $t = 9$ and $t = 12$, units most active during these intervals are consistently assigned lower indices. This assignment is not merely cosmetic: it embeds operational meaning directly into the labeling, aligning the mathematical representation with the merit-order logic used in practice. A similar principle applies at the plant level, where the package committing its units earlier and sustaining them longer is labeled Package 1, while Package 2 provides complementary capacity through the delayed activation of GT 3.

A limitation arises when additional inter-temporal constraints, such as ramping, are introduced. In such settings, unit interchangeability may be partially broken by differences in operational histories, reducing the effective symmetry group and requiring a more refined treatment of admissible permutations. Within the present formulation, however, the symmetry-breaking approach provides a compact and interpretable representation of the solution space, where unit indices directly reflect their operational role during the most critical periods of system operation.